## Higgs Boson Machine Learning Challenge
### Version Colab: descarga inicial de datos

Este notebook esta pensado para ejecutarse en Google Colab. Su unico objetivo es configurar Kaggle, descargar los datos de la competencia `higgs-boson` y dejarlos disponibles para los siguientes pasos.

## Configuracion de Kaggle

La siguiente celda solicita el archivo `kaggle.json`, lo instala en `~/.kaggle/`, configura permisos y deja preparada la carpeta `data/higgs-boson/`.

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
import zipfile
from pathlib import Path

from google.colab import files

uploaded = files.upload()

if 'kaggle.json' not in uploaded:
    raise FileNotFoundError('Debe subir el archivo kaggle.json para continuar.')

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_file = kaggle_dir / 'kaggle.json'
kaggle_file.write_bytes(uploaded['kaggle.json'])
kaggle_file.chmod(0o600)

os.environ['KAGGLE_CONFIG_DIR'] = str(kaggle_dir)

data_dir = Path.cwd() / 'data' / 'higgs-boson'
data_dir.mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', 'kaggle'], check=True)

print(f'KAGGLE_CONFIG_DIR = {os.environ["KAGGLE_CONFIG_DIR"]}')
print(f'data_dir = {data_dir.resolve()}')

## Descarga y descompresion

In [ ]:
archive_path = data_dir / 'higgs-boson.zip'

subprocess.run(
    [
        'kaggle',
        'competitions',
        'download',
        '-c',
        'higgs-boson',
        '-p',
        str(data_dir),
    ],
    check=True,
)

with zipfile.ZipFile(archive_path, 'r') as zip_ref:
    zip_ref.extractall(data_dir)

for zip_path in sorted(data_dir.glob('*.zip')):
    if zip_path.name == archive_path.name:
        continue
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)

print('Descarga y descompresion finalizadas.')

## Verificacion

In [ ]:
required_files = [
    'training.csv',
    'test.csv',
    'random_submission.csv',
    'HiggsBosonCompetition_AMSMetric_rev1.py',
]

missing_files = [name for name in required_files if not (data_dir / name).exists()]

if missing_files:
    raise FileNotFoundError(
        'Faltan archivos esperados luego de la descarga: ' + ', '.join(missing_files)
    )

print('Archivos principales disponibles:')
for name in required_files:
    print(f'- {data_dir / name}')